***

# **EJ Scripting**

***

In [357]:
# Packages
import pandas as pd 
import numpy as np
import os
from pathlib import Path
import re
from datetime import datetime

# Data
df = pd.read_excel("I:\\Projects\\Warren\\Environmental_Justice_March_2023\\SACOG_EJ_UPDATE_2024\\Python\\YOUR_OUTPUT_FOLDER\\Block Groups Processed - Ready for Calculator.xlsx", sheet_name = "ready_to_calculate")


***

## **First Step: Calculator**

***

In [26]:
# Renaming some df columns to match the current code
rename_dict = {
    'Minority_Total': 'Minority_Estimate!!Total:',
    'LowIncome_Total': 'LowIncome_Estimate!!Total:',
    'Older75Plus_Total': 'Older75plus_Estimate!!Total:',
    'Linguistically_Total': 'Linguistically_Estimate!!Total:',
    'SingleParent_Total': 'SingleParent_Estimate!!Total:',
    'EducationAttainment_Total': 'EducationAttainment_Estimate!!Total:',
    'GrossRent_Total': 'GrossRent_Estimate!!Total:',
    'GrossMortgage_Total': 'GrossMortgage_Estimate!!Total:',
    'Disability_Total': 'Disability_Estimate!!Total:'
}

# Used a dict just to keep track of what I renamed too
df = df.rename(columns=rename_dict)

In [311]:
# Idea: There are a lot of columns in the df that follow a similar naming schema, total at the end, count as well. 
# Let's take the beginning name, assign the num + denom, and then divide by the corresponding count and total 
# Can then loop the function through to make it so that we get every classifier
# test = df # doing this so we don't have four columns when we go for the actual test. This is for trial run

def calculate_percentage(df, prefix, quantile_val=0.75):
    
    marked_cols = df.filter(like=prefix).columns.tolist()
    # print(marked_cols)

    total_col = [col for col in marked_cols if col.endswith('Total')]
    non_total_col = [col for col in marked_cols if not col.endswith('Total')]
    # print(total_col, non_total_col)

    
    if total_col and non_total_col:
        total_col = total_col[0] # This should only be a list of 1 val, but must extract
        non_total_col = non_total_col[0] # This should only be a list of 1 val, but must extract
    else:
        raise ValueError(f"Could not find total and non-total columns for prefix: {prefix}.")
    

    percent_col = f"{prefix}_Percent"
    # print(percent_col)

    # I just copied this step from Warren's above
    df[percent_col] = (df[non_total_col].fillna(0) / df[total_col].fillna(1)) * 100
    threshold = df[percent_col].dropna().quantile(quantile_val)
    print(f"Threshold for", prefix, "is:", threshold)
    df[f"{prefix}_Marked"] = (df[percent_col] >= threshold).astype(int)

    return df

prefixes = set([col.split('_')[0] for col in list(df.columns[8:])])
print(prefixes)

for i in prefixes:
    calculate_percentage(df, prefix=i, quantile_val=0.75)

display(df.head(3))

{'Disability', 'Linguistically', 'EducationAttainment', 'LowIncome', 'Older75Plus', 'GrossMortgage', 'Minority', 'SingleParent', 'GrossRent'}
Threshold for Disability is: 6.64885558456088
Threshold for Linguistically is: 7.51708164148649
Threshold for EducationAttainment is: 15.536418499043977
Threshold for LowIncome is: 87.83642581247369
Threshold for Older75Plus is: 9.114472453678758
Threshold for GrossMortgage is: 15.232295406710413
Threshold for Minority is: 71.82798170466577
Threshold for SingleParent is: 14.570803040317251
Threshold for GrossRent is: 35.26406926406926


,State FIPS,County FIPS,County Name,Tract ID,Block Group ID,NAME,Year,Race_Ethnicity,Disability_Disability,Disability_Total,...,Older75Plus_Percent,Older75Plus_Marked,GrossMortgage_Percent,GrossMortgage_Marked,Minority_Percent,Minority_Marked,SingleParent_Percent,SingleParent_Marked,GrossRent_Percent,GrossRent_Marked
0,6,17,El Dorado,30201,1,Block Group 1; Census Tract 302.01; El Dorado ...,2022,All,18,316,...,0.00000,0,0.00000,0,66.578600,0,11.392405,0,55.778894,1
1,6,17,El Dorado,30201,2,Block Group 2; Census Tract 302.01; El Dorado ...,2022,All,25,383,...,6.82243,0,0.00000,0,69.813084,0,7.049608,0,6.849315,0
2,6,17,El Dorado,30201,3,Block Group 3; Census Tract 302.01; El Dorado ...,2022,All,0,234,...,0.00000,0,10.25641,0,39.325843,0,11.111111,0,36.752137,1


In [312]:
df['Minority_Percent'] = 100 - df['Minority_Percent']

threshold = df['Minority_Percent'].dropna().quantile(0.75)

df['Minority_Marked'] = (df['Minority_Marked'] >= threshold).astype(int)

***

## **Second Step: Criteria**

***

In [ ]:
# Warren's
def calculate_additional_criteria(input_directory, output_directory, year):
    # Ensure the output directory exists
    output_directory_path = Path(output_directory)
    output_directory_path.mkdir(parents=True, exist_ok=True)

    # Define the input file path using the year variable
    input_file_name = f"SACOG_EJ_{year}_Regional_Raw_Calculated.csv"
    input_file_path = Path(input_directory) / input_file_name

    # Load the data
    print(f"Processing: {input_file_path}")
    df = pd.read_csv(input_file_path)

    # List of categories to be marked
    category_columns = [col for col in base_df.columns if '_Marked' in col]

    # Calculate Other factors Summed excluding Minority and Low_Income
    other_factors_columns = [col for col in category_columns if col not in ['Minority_Marked', 'Low_Income_Marked']]
    base_df['Other_factors_Summed'] = base_df[other_factors_columns].sum(axis=1)

    # Calculate identified categories and all categories summed
    base_df['Minority_identified'] = (base_df['Minority_Marked'] == 1).astype(int)
    base_df['Low_Income_identified'] = (base_df['Low_Income_Marked'] == 1).astype(int)
    base_df['Other_factors_identified'] = (base_df['Other_factors_Summed'] >= 4).astype(int)#
    base_df['All_categories_summed'] = base_df[category_columns].sum(axis=1)
    base_df['All_identified_summed'] = base_df[['Minority_identified', 'Low_Income_identified', 'Other_factors_identified']].sum(axis=1)

    # Add another field for tract/blockgroup to join with feature class
    base_df['Boundary_geo_join'] = base_df['Geography'].str[-11:]

    # Save the modified DataFrame
    output_file_name = f"SACOG_EJ_{year}_Regional_Criteria.csv"
    output_file_path = output_directory_path / output_file_name
    base_df.to_csv(output_file_path, index=False)
    print(f"Processed and saved: {output_file_path}")

# Set the year for which you want to process the data
year = '2022'
input_directory = r'I:\Projects\Warren\Environmental_Justice_March_2023\SACOG_EJ_UPDATE_2024\EJ_2022\Calculator'
output_directory = r'I:\Projects\Warren\Environmental_Justice_March_2023\SACOG_EJ_UPDATE_2024\EJ_2022\Criteria'

calculate_additional_criteria(input_directory, output_directory, year)
print('All done...^_^..V')


In [315]:
# Let's add on to og func. 

def ej_labeling(df, prefix, quantile_val=0.75):
    
    ################### Making percents and marking #######################################
    marked_cols = df.filter(like=prefix).columns.tolist()
    # print(marked_cols)

    total_col = [col for col in marked_cols if col.endswith('Total')]
    non_total_col = [col for col in marked_cols if not col.endswith('Total')]
    # print(total_col, non_total_col)

    
    if total_col and non_total_col:
        total_col = total_col[0] # This should only be a list of 1 val, but must extract
        non_total_col = non_total_col[0] # This should only be a list of 1 val, but must extract
    else:
        raise ValueError(f"Could not find total and non-total columns for prefix: {prefix}.")
    

    percent_col = f"{prefix}_Percent"
    # print(percent_col)

    # I just copied this step from Warren's above
    df[percent_col] = (df[non_total_col].fillna(0) / df[total_col].fillna(1)) * 100
    threshold = df[percent_col].dropna().quantile(quantile_val)
    print(f"Threshold for", prefix, "is: ", threshold)
    df[f"{prefix}_Marked"] = (df[percent_col] >= threshold).astype(int)

    ################### Labeling #######################################

    # Same as Warren's
    category_columns = [col for col in df.columns if '_Marked' in col]

    # Just doing it a lil more dynamically,
    for col in category_columns:
        col_base = col.replace('_Marked', '_identified')
        df[col_base] = (df[col] == 1).astype(int)
    
    # Warren's
    other_factors_columns = [col for col in category_columns if col not in ['Minority_Marked', 'Low_Income_Marked']]
    df['Other_factors_Summed'] = df[other_factors_columns].sum(axis=1)

    # Warrens
    df['Other_factors_identified'] = (df['Other_factors_Summed'] >= 4).astype(int)

    # Summing
    identified_columns = [col for col in df.columns if '_identified' in col]
    df['All_categories_summed'] = df[category_columns].sum(axis=1)
    df['All_identified_summed'] = df[identified_columns].sum(axis=1)

    return df

prefixes = set([col.split('_')[0] for col in list(df.columns[8:])])
print(prefixes)

for i in prefixes:
    ej_labeling(df, prefix=i, quantile_val=0.75)

# Export

# One column for all of the block group id
# + All of the marked columns
# Josh wants the geography field from the CES4 file

{'Disability', 'Linguistically', 'EducationAttainment', 'LowIncome', 'Older75Plus', 'GrossMortgage', 'Minority', 'SingleParent', 'GrossRent'}
Threshold for Disability is:  6.64885558456088
Threshold for Linguistically is:  7.51708164148649
Threshold for EducationAttainment is:  15.536418499043977
Threshold for LowIncome is:  87.83642581247369
Threshold for Older75Plus is:  9.114472453678758
Threshold for GrossMortgage is:  15.232295406710413
Threshold for Minority is:  71.82798170466577
Threshold for SingleParent is:  14.570803040317251
Threshold for GrossRent is:  35.26406926406926


***

## **Third Step: Label**

***

In [ ]:
# For CES4, need to run the process for arcpro
# Put a print statement in there about like hey it's been more six months it's 
# SACOG Criteria file to YAML

In [471]:
# Final function process.

def ej_process(input_directory, output_directory, quantile_val):

    # Checking year to see if we should re-run code--nothing too crazy 
    year_match = re.search(r'(\d{4})', str(input_directory))
    year = year_match.group(1) if year_match else 'UnknownYear'
    
    # input_file_path = input_directory / f'SACOG_{year}_Criteria_wCES.csv'
    
    current_date = datetime.now()
    six_months_ago = current_date - pd.DateOffset(months=6)

    year_date = datetime(int(year), 1, 1)

    if year_date < six_months_ago:
        print(f"Warning: The given year is more than 6 months older than the current date ({current_date.strftime('%Y-%m-%d')}). User might want to re-run the Census script!")

    ############################################# Calculate ##############################################
    df = pd.read_excel(input_directory)

    df['HousingBurden'] = (df['GrossMortgage_50 pct or more'] + df['GrossRent_50 pct or more'])
    df['HousingBurden_Total'] = (df['GrossMortgage_Total'] + df['GrossRent_Total'])

    # This might have to change depending on the structure of the input dataframe.
    prefixes = set([col.split('_')[0] for col in list(df.columns[8:])])

    for prefix in prefixes:    
        marked_cols = df.filter(like=prefix).columns.tolist()

        total_col = [col for col in marked_cols if col.endswith('Total')]
        non_total_col = [col for col in marked_cols if not col.endswith('Total')]

        if total_col and non_total_col:
            total_col = total_col[0] 
            non_total_col = non_total_col[0] 
        else:
            raise ValueError(f"Could not find total and non-total columns for prefix: {prefix}.")
        
        percent_col = f"{prefix}_Percent"

        df[percent_col] = (df[non_total_col].fillna(0) / df[total_col].fillna(1)) * 100
        threshold = df[percent_col].dropna().quantile(quantile_val)
        print(f"Threshold for", prefix, "is: ", threshold)
        # df[f"{prefix}_Marked"] = (df[percent_col] >= threshold).astype(int)
        df[f"{prefix}_Marked"] = 0
        df.loc[df[percent_col] > threshold, f"{prefix}_Marked"] = 1

    df['Minority_Percent'] = 100 - df['Minority_Percent']
    threshold = df['Minority_Percent'].dropna().quantile(0.75) # in Second calculator script we used 0.7
    df["Minority_Marked"] = 0
    df.loc[df["Minority_Percent"] >= threshold, "Minority_Marked"] = 1

    df['LowIncome_Percent'] = 100 - df['LowIncome_Percent']
    threshold = df['LowIncome_Percent'].dropna().quantile(0.75) # in second calculator script we used 0.45
    df["LowIncome_Marked"] = 0
    df.loc[df["LowIncome_Percent"] >= threshold, "LowIncome_Marked"] = 1
    
    ################### Labelling #######################################

    category_columns = [col for col in df.columns if '_Marked' in col]

    for col in category_columns:
        col_base = col.replace('_Marked', '_identified')
        df[col_base] = (df[col] == 1).astype(int)
    
    other_factors_columns = [col for col in category_columns if col not in ['Minority_Marked', 'Low_Income_Marked']]
    df['Other_factors_Summed'] = df[other_factors_columns].sum(axis=1)

    df['Other_factors_identified'] = (df['Other_factors_Summed'] >= 4).astype(int)

    identified_columns = [col for col in df.columns if '_identified' in col]
    df['All_categories_summed'] = df[category_columns].sum(axis=1)
    df['All_identified_summed'] = df[identified_columns].sum(axis=1)

    ############################### Final Criteria #####################################

    # Define the categories as before
    categories = [
        ('Highest Priority', ['Minority_identified', 'LowIncome_identified', 'Other_factors_identified', 'CES4_designated']),
        ('LowInc/Min/Oth', ['Minority_identified', 'LowIncome_identified', 'Other_factors_identified']),
        ('LowInc/Min/CES', ['Minority_identified', 'LowIncome_identified', 'CES4_designated']),
        ('LowInc/Min', ['Minority_identified', 'LowIncome_identified']),
        ('LowInc/Oth', ['LowIncome_identified', 'Other_factors_identified']),
        ('Min/Oth', ['Minority_identified', 'Other_factors_identified']),
        ('LowInc or Min and CES', ['LowIncome_identified', 'Minority_identified', 'CES4_designated']),  # Adjusted case
        ('Minority', ['Minority_identified']),
        ('Low Income', ['LowIncome_identified']),
        ('Other factors', ['Other_factors_identified']),
        ('CES4', ['CES4_designated'])
    ]

    # Making the categories, and then converting to wide. 
    for category, factors in categories:
        df[category] = 0

        if category == 'LowInc or Min and CES':
            # In the code, there was a special case for this one where it had to be an or, this is just to catch this correctly
            condition = ((df['LowIncome_identified'] == 1) | (df['Minority_identified'] == 1)) & (df['CES4_designated'] == 1)
        else:
            # General case: all factors must be 1 in order to be labelled
            condition = np.all([df[factor] == 1 for factor in factors], axis=0)
        
        # Assign 1 where the condition is True
        df.loc[condition, category] = 1

        # Define conditions and choices for labeling
    conditions = [
        (df['Minority_identified'] == 1) & (df['LowIncome_identified'] == 1) & (df['Other_factors_identified'] == 1) & (df['CES4_designated'] == 1), # Highest Priority
        (df['LowIncome_identified'] == 1) & (df['Minority_identified'] == 1) & (df['Other_factors_identified'] == 1), # LowInc/Min/Oth
        (df['LowIncome_identified'] == 1) & (df['Minority_identified'] == 1) & (df['CES4_designated'] == 1), # LowInc/Min/CES
        (df['LowIncome_identified'] == 1) & (df['Minority_identified'] == 1), # LowInc/Min
        (df['LowIncome_identified'] == 1) & (df['Other_factors_identified'] == 1), # LowInc/Oth
        (df['Minority_identified'] == 1) & (df['Other_factors_identified'] == 1), # Min/Oth
        ((df['LowIncome_identified'] == 1) | (df['Minority_identified'] == 1)) & (df['CES4_designated'] == 1), # LowInc or Min and CES
        (df['Minority_identified'] == 1) & (df['LowIncome_identified'] != 1) & (df['Other_factors_identified'] != 1) & (df['CES4_designated'] != 1), # Minority
        (df['Minority_identified'] != 1) & (df['LowIncome_identified'] == 1) & (df['Other_factors_identified'] != 1) & (df['CES4_designated'] != 1), # Low Income
        (df['Minority_identified'] != 1) & (df['LowIncome_identified'] != 1) & (df['Other_factors_identified'] == 1) & (df['CES4_designated'] != 1), # Other factors
        (df['Minority_identified'] != 1) & (df['LowIncome_identified'] != 1) & (df['Other_factors_identified'] != 1) & (df['CES4_designated'] == 1) # CES4
    ]
    choices = ['Highest Priority', 'LowInc/Min/Oth', 'LowInc/Min/CES', 'LowInc/Min', 'LowInc/Oth', 'Min/Oth',
                'LowInc or Min and CES', 'Minority', 'Low Income', 'Other factors', 'CES4']

    # Apply conditions to assign labels
    df['EJ_Label'] = np.select(conditions, choices, default='')
    # Now we subset the dataframe to only keep the metadata cols and the condition columns
    # meta_cols = df.columns[:6] + list('Geographic Area Name')
    meta_list = ['Geography', 'Geographic Area Name', 'State FIPS', 'County FIPS', 
                 'County Name', 'Tract ID', 'Block Group ID', 'Year']
    meta_cols = [col for col in meta_list if col in df.columns]
    cat_cols = [category[0] for category in categories]
    wish_cols = list(meta_cols) + list('Geography', 'Geographic Area Name') + cat_cols
    
    # If the wish cols are found in the df, we will keep em. If not, we proceed without.
    keep_cols = [col for col in wish_cols if col in df.columns]

    # Subsetting dataframe now
    df = df[keep_cols]

    # Export ready!
    output_file_csv = Path(output_directory) / f"SACOG_EJ_{year}_Labelled.csv"
    output_file_xlsx = Path(output_directory) / f"SACOG_EJ_{year}_Labelled.xlsx"
    df.to_csv(output_file_csv, index=False)
    df.to_excel(output_file_xlsx, index=False)

    print(f"Output saved to {output_file_csv} and {output_file_xlsx}, all done <^_^>")

input_directory = Path('I:/Projects/Warren/Environmental_Justice_March_2023/SACOG_EJ_UPDATE_2024/Python/YOUR_OUTPUT_FOLDER/Block Groups Processed - Ready for Calculator.xlsx') # can just make it so that whatever this is just gets read in
output_directory = Path("C:/Users/jchoy/Documents/Python Projects/Environmental Justice")

ej_process(input_directory, output_directory, quantile_val=0.75)

***

## **Test Run!**

***

In [466]:
# Calm luh test run
# Need to use the criteria dataset, and the original. 
# No exports on this one. 
# Data
df = pd.read_excel("I:\\Projects\\Warren\\Environmental_Justice_March_2023\\SACOG_EJ_UPDATE_2024\\Python\\YOUR_OUTPUT_FOLDER\\Block Groups Processed - Ready for Calculator.xlsx", sheet_name = "ready_to_calculate")

input_directory = Path('I:/Projects/Warren/Environmental_Justice_March_2023/SACOG_EJ_UPDATE_2024/EJ_2022/CES4')
year = '2022'
input_file = input_directory / f'SACOG_{year}_Criteria_wCES.csv' # Adjust based on actual input file naming
criteria = pd.read_csv(input_file)


In [467]:
merger = df.merge(criteria[['Geography', 'Geographic Area Name', 'CES4_designated']], how = 'left',
                left_on = 'NAME', right_on = 'Geographic Area Name').drop(columns = ['NAME'])

merger = merger.dropna()

# Specify the columns you want to move to the left
columns_to_move = ['Geography', 'Geographic Area Name', 'CES4_designated']

# Create a new column order
new_column_order = columns_to_move + [col for col in merger.columns if col not in columns_to_move]

# Reorder the DataFrame
merger = merger[new_column_order]

In [468]:
merger['HousingBurden'] = (merger['GrossMortgage_50 pct or more'] + merger['GrossRent_50 pct or more'])

merger['HousingBurden_Total'] = (merger['GrossMortgage_Total'] + merger['GrossRent_Total'])

merger = merger.drop(['GrossMortgage_Total', 'GrossRent_Total', 'GrossMortgage_50 pct or more', 'GrossRent_50 pct or more'], axis=1)

In [469]:
def test_ej(df, quantile_val):
    # This might have to change depending on the structure of the input dataframe.
    prefixes = set([col.split('_')[0] for col in list(df.columns[10:])])
    print(prefixes)

    for prefix in prefixes:    
        marked_cols = df.filter(like=prefix).columns.tolist()

        total_col = [col for col in marked_cols if col.endswith('Total')]
        non_total_col = [col for col in marked_cols if not col.endswith('Total')]

        if total_col and non_total_col:
            total_col = total_col[0] 
            non_total_col = non_total_col[0] 
        else:
            raise ValueError(f"Could not find total and non-total columns for prefix: {prefix}.")
        
        percent_col = f"{prefix}_Percent"

        df[percent_col] = (df[non_total_col].fillna(0) / df[total_col].fillna(1)) * 100
        threshold = df[percent_col].dropna().quantile(quantile_val)
        print(f"Threshold for", prefix, "is: ", threshold)
        # df[f"{prefix}_Marked"] = (df[percent_col] >= threshold).astype(int)
        df[f"{prefix}_Marked"] = 0
        df.loc[df[percent_col] >= threshold, f"{prefix}_Marked"] = 1

    df['Minority_Percent'] = 100 - df['Minority_Percent']
    threshold = df['Minority_Percent'].dropna().quantile(0.75) # in Second calculator script we used 0.7
    df["Minority_Marked"] = 0
    df.loc[df["Minority_Percent"] >= threshold, "Minority_Marked"] = 1

    df['LowIncome_Percent'] = 100 - df['LowIncome_Percent']
    threshold = df['LowIncome_Percent'].dropna().quantile(0.75) # in second calculator script we used 0.45
    df["LowIncome_Marked"] = 0
    df.loc[df["LowIncome_Percent"] >= threshold, "LowIncome_Marked"] = 1

    ################### Labelling #######################################

    category_columns = [col for col in df.columns if '_Marked' in col]

    for col in category_columns:
        col_base = col.replace('_Marked', '_identified')
        df[col_base] = (df[col] == 1).astype(int)
    
    other_factors_columns = [col for col in category_columns if col not in ['Minority_Marked', 'Low_Income_Marked']]
    df['Other_factors_Summed'] = df[other_factors_columns].sum(axis=1)

    print(df['Other_factors_Summed'].value_counts())

    df['Other_factors_identified'] = (df['Other_factors_Summed'] >= 4).astype(int)

    identified_columns = [col for col in df.columns if '_identified' in col]
    df['All_categories_summed'] = df[category_columns].sum(axis=1)
    df['All_identified_summed'] = df[identified_columns].sum(axis=1)

    ############################### Final Criteria #####################################

    # Define the categories as before
    categories = [
        ('Highest Priority', ['Minority_identified', 'LowIncome_identified', 'Other_factors_identified', 'CES4_designated']),
        ('LowInc/Min/Oth', ['Minority_identified', 'LowIncome_identified', 'Other_factors_identified']),
        ('LowInc/Min/CES', ['Minority_identified', 'LowIncome_identified', 'CES4_designated']),
        ('LowInc/Min', ['Minority_identified', 'LowIncome_identified']),
        ('LowInc/Oth', ['LowIncome_identified', 'Other_factors_identified']),
        ('Min/Oth', ['Minority_identified', 'Other_factors_identified']),
        ('LowInc or Min and CES', ['LowIncome_identified', 'Minority_identified', 'CES4_designated']),  # Adjusted case
        ('Minority', ['Minority_identified']),
        ('Low Income', ['LowIncome_identified']),
        ('Other factors', ['Other_factors_identified']),
        ('CES4', ['CES4_designated'])
    ]

    # Making the categories, and then converting to wide. 
    for category, factors in categories:
        df[category] = 0

        if category == 'LowInc or Min and CES':
            # In the code, there was a special case for this one where it had to be an or, this is just to catch this correctly
            condition = ((df['LowIncome_identified'] == 1) | (df['Minority_identified'] == 1)) & (df['CES4_designated'] == 1)
        else:
            # General case: all factors must be 1 in order to be labelled
            condition = np.all([df[factor] == 1 for factor in factors], axis=0)
        
        # Assign 1 where the condition is True
        df.loc[condition, category] = 1

        # Define conditions and choices for labeling
    conditions = [
        (df['Minority_identified'] == 1) & (df['LowIncome_identified'] == 1) & (df['Other_factors_identified'] == 1) & (df['CES4_designated'] == 1), # Highest Priority
        (df['LowIncome_identified'] == 1) & (df['Minority_identified'] == 1) & (df['Other_factors_identified'] == 1), # LowInc/Min/Oth
        (df['LowIncome_identified'] == 1) & (df['Minority_identified'] == 1) & (df['CES4_designated'] == 1), # LowInc/Min/CES
        (df['LowIncome_identified'] == 1) & (df['Minority_identified'] == 1), # LowInc/Min
        (df['LowIncome_identified'] == 1) & (df['Other_factors_identified'] == 1), # LowInc/Oth
        (df['Minority_identified'] == 1) & (df['Other_factors_identified'] == 1), # Min/Oth
        ((df['LowIncome_identified'] == 1) | (df['Minority_identified'] == 1)) & (df['CES4_designated'] == 1), # LowInc or Min and CES
        (df['Minority_identified'] == 1) & (df['LowIncome_identified'] != 1) & (df['Other_factors_identified'] != 1) & (df['CES4_designated'] != 1), # Minority
        (df['Minority_identified'] != 1) & (df['LowIncome_identified'] == 1) & (df['Other_factors_identified'] != 1) & (df['CES4_designated'] != 1), # Low Income
        (df['Minority_identified'] != 1) & (df['LowIncome_identified'] != 1) & (df['Other_factors_identified'] == 1) & (df['CES4_designated'] != 1), # Other factors
        (df['Minority_identified'] != 1) & (df['LowIncome_identified'] != 1) & (df['Other_factors_identified'] != 1) & (df['CES4_designated'] == 1) # CES4
    ]
    choices = ['Highest Priority', 'LowInc/Min/Oth', 'LowInc/Min/CES', 'LowInc/Min', 'LowInc/Oth', 'Min/Oth',
                'LowInc or Min and CES', 'Minority', 'Low Income', 'Other factors', 'CES4']

    # Apply conditions to assign labels
    df['EJ_Label'] = np.select(conditions, choices, default='')

    # Now we subset the dataframe to only keep the first six columns and the condition columns
    # meta_list = ['Geography', 'Geographic Area Name', 'State FIPS', 'County FIPS', 
    #              'County Name', 'Tract ID', 'Block Group ID', 'Year']
    # meta_cols = [col for col in meta_list if col in df.columns]
    # cat_cols = [category[0] for category in categories]
    # wish_cols = meta_cols + cat_cols
    
    # # If the wish cols are found in the df, we will keep em. If not, we proceed without.
    # keep_cols = list(dict.fromkeys([col for col in wish_cols if col in df.columns]))

    # # Subsetting dataframe now
    # df = df[keep_cols]

    return df

final_frame = test_ej(merger, quantile_val= 0.75)

{'Disability', 'Linguistically', 'EducationAttainment', 'Older75Plus', 'Minority', 'SingleParent', 'HousingBurden', 'LowIncome'}
Threshold for Disability is:  6.87960687960688
Threshold for Linguistically is:  7.801418439716312
Threshold for EducationAttainment is:  16.12576064908722
Threshold for Older75Plus is:  9.10144927536232
Threshold for Minority is:  70.6980880391285
Threshold for SingleParent is:  14.841849148418493
Threshold for HousingBurden is:  22.19607843137255
Threshold for LowIncome is:  87.08108108108108
Other_factors_Summed
1    434
0    408
2    274
3    178
4    130
5     90
6     36
7      3
Name: count, dtype: int64


In [460]:
display(final_frame[final_frame['Highest Priority']==1])

display(final_frame[(final_frame['Geography'].duplicated())])



,Geography,Geographic Area Name,CES4_designated,State FIPS,County FIPS,County Name,Tract ID,Block Group ID,Year,Race_Ethnicity,Disability_Disability,Disability_Total,EducationAttainment_Less than high school,EducationAttainment_Total,Linguistically_Limited english speaking household,Linguistically_Total,LowIncome_2 and over,LowIncome_Total,Minority_Total,Minority_White (NH),Older75Plus_75Plus,Older75Plus_Total,SingleParent_No Spouse,SingleParent_Total,HousingBurden,HousingBurden_Total,Disability_Percent,Disability_Marked,Linguistically_Percent,Linguistically_Marked,EducationAttainment_Percent,EducationAttainment_Marked,Older75Plus_Percent,Older75Plus_Marked,Minority_Percent,Minority_Marked,SingleParent_Percent,SingleParent_Marked,HousingBurden_Percent,HousingBurden_Marked,LowIncome_Percent,LowIncome_Marked,Disability_identified,Linguistically_identified,EducationAttainment_identified,Older75Plus_identified,Minority_identified,SingleParent_identified,HousingBurden_identified,LowIncome_identified,Other_factors_Summed,Other_factors_identified,All_categories_summed,All_identified_summed,Highest Priority,LowInc/Min/Oth,LowInc/Min/CES,LowInc/Min,LowInc/Oth,Min/Oth,LowInc or Min and CES,Minority,Low Income,Other factors,CES4
428,1500000US060670007001,Block Group 1; Census Tract 7; Sacramento Coun...,1.0,6,67,Sacramento,700,1,2022,All,178,595,619,2364,257,595,168,803,2614,821,303,2614,0,595,111,595,29.915966,1,43.193277,1,26.184433,1,11.591431,1,68.592196,1,0.000000,0,18.655462,0,79.078456,1,1,1,1,1,1,0,0,1,5,1,6,7,1,1,1,1,1,1,1,1,1,1,1
431,1500000US060670011021,Block Group 1; Census Tract 11.02; Sacramento ...,1.0,6,67,Sacramento,1102,1,2022,All,50,415,111,521,17,415,363,735,735,249,18,735,32,415,186,415,12.048193,1,4.096386,0,21.305182,1,2.448980,0,66.122449,1,7.710843,0,44.819277,1,50.612245,1,1,0,1,0,1,0,1,1,4,1,5,6,1,1,1,1,1,1,1,1,1,1,1
515,1500000US060670032021,Block Group 1; Census Tract 32.02; Sacramento ...,1.0,6,67,Sacramento,3202,1,2022,All,0,439,169,761,9,439,794,1176,1176,334,34,1176,79,439,112,439,0.000000,0,2.050114,0,22.207622,1,2.891156,0,71.598639,1,17.995444,1,25.512528,1,32.482993,1,0,0,1,0,1,1,1,1,4,1,5,6,1,1,1,1,1,1,1,1,1,1,1
516,1500000US060670032022,Block Group 2; Census Tract 32.02; Sacramento ...,1.0,6,67,Sacramento,3202,2,2022,All,42,365,147,869,58,365,473,1187,1233,197,129,1233,82,365,77,365,11.506849,1,15.890411,1,16.915995,1,10.462287,1,84.022709,1,22.465753,1,21.095890,0,60.151643,1,1,1,1,1,1,1,0,1,6,1,7,8,1,1,1,1,1,1,1,1,1,1,1
517,1500000US060670032023,Block Group 3; Census Tract 32.02; Sacramento ...,1.0,6,67,Sacramento,3202,3,2022,All,56,525,422,934,195,525,374,1346,1346,101,143,1346,62,525,186,525,10.666667,1,37.142857,1,45.182013,1,10.624071,1,92.496285,1,11.809524,0,35.428571,1,72.213967,1,1,1,1,1,1,0,1,1,6,1,7,8,1,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1513,1500000US061130102033,Block Group 3; Census Tract 102.03; Yolo Count...,1.0,6,113,Yolo,10203,3,2022,All,147,692,441,1370,142,692,566,2250,2355,557,90,2355,181,692,283,692,21.242775,1,20.520231,1,32.189781,1,3.821656,0,76.348195,1,26.156069,1,40.895954,1,74.844444,1,1,1,1,0,1,1,1,1,6,1,7,8,1,1,1,1,1,1,1,1,1,1,1
1515,1500000US061130102041,Block Group 1; Census Tract 102.04; Yolo Count...,1.0,6,113,Yolo,10204,1,2022,All,0,612,353,855,17,612,520,1598,1598,267,53,1598,173,612,236,612,0.000000,0,2.777778,0,41.286550,1,3.316646,0,83.291615,1,28.267974,1,38.562092,1,67.459324,1,0,0,1,0,1,1,1,1,4,1,5,6,1,1,1,1,1,1,1,1,1,1,1
1656,1500000US061150404001,Block Group 1; Census Tract 404; Yuba County; ...,1.0,6,115,Yuba,40400,1,2022,All,178,765,653,1634,76,765,1577,3025,3025,992,135,3025,104,765,116,765,23.267974,1,9.934641,1,39.963280,1,4.462810,0,67.206612,1,13.594771,0,15.163399,0,47.867769,1,1,1,1,0,1,0,0,1,4,1,5,6

,Geography,Geographic Area Name,CES4_designated,State FIPS,County FIPS,County Name,Tract ID,Block Group ID,Year,Race_Ethnicity,Disability_Disability,Disability_Total,EducationAttainment_Less than high school,EducationAttainment_Total,Linguistically_Limited english speaking household,Linguistically_Total,LowIncome_2 and over,LowIncome_Total,Minority_Total,Minority_White (NH),Older75Plus_75Plus,Older75Plus_Total,SingleParent_No Spouse,SingleParent_Total,HousingBurden,HousingBurden_Total,Disability_Percent,Disability_Marked,Linguistically_Percent,Linguistically_Marked,EducationAttainment_Percent,EducationAttainment_Marked,Older75Plus_Percent,Older75Plus_Marked,Minority_Percent,Minority_Marked,SingleParent_Percent,SingleParent_Marked,HousingBurden_Percent,HousingBurden_Marked,LowIncome_Percent,LowIncome_Marked,Disability_identified,Linguistically_identified,EducationAttainment_identified,Older75Plus_identified,Minority_identified,SingleParent_identified,HousingBurden_identified,LowIncome_identified,Other_factors_Summed,Other_factors_identified,All_categories_summed,All_identified_summed,Highest Priority,LowInc/Min/Oth,LowInc/Min/CES,LowInc/Min,LowInc/Oth,Min/Oth,LowInc or Min and CES,Minority,Low Income,Other factors,CES4


***

## **NULL SPACE**

***

In [234]:
# # import SACOG_Criteria file

# # We will run this by merging the two files together and taking the important columns.
# input_directory = Path('I:/Projects/Warren/Environmental_Justice_March_2023/SACOG_EJ_UPDATE_2024/EJ_2022/CES4')
# year = '2022'
# input_file = input_directory / f'SACOG_{year}_Criteria_wCES.csv' # Adjust based on actual input file naming
# criteria = pd.read_csv(input_file)

# ooo the mismatching shapes actually makes it a little questionable to combine these dataframes together. the shapes r diff so we got some missing ones. 

# display(criteria.shape)
# display(df.shape)

# merger = df.merge(criteria[['Geography', 'Geographic Area Name', 'CES4_designated']], how = 'left',
#                 left_on = 'NAME', right_on = 'Geographic Area Name').drop(columns = ['NAME'])

In [88]:
# # List of categories to be marked

# # Trying this with my dataframe from step 1

# category_columns = [col for col in df.columns if '_Marked' in col]

# # Calculate Other factors Summed excluding Minority and Low_Income
# other_factors_columns = [col for col in category_columns if col not in ['Minority_Marked', 'Low_Income_Marked']]
# df['Other_factors_Summed'] = df[other_factors_columns].sum(axis=1)

# # Calculate identified categories and all categories summed
# df['Minority_identified'] = (df['Minority_Marked'] == 1).astype(int)
# df['LowIncome_identified'] = (df['LowIncome_Marked'] == 1).astype(int)
# df['Other_factors_identified'] = (df['Other_factors_Summed'] >= 4).astype(int)#
# df['All_categories_summed'] = df[category_columns].sum(axis=1)
# df['All_identified_summed'] = df[['Minority_identified', 'LowIncome_identified', 'Other_factors_identified']].sum(axis=1)

# # Add another field for tract/blockgroup to join with feature class
# df['Boundary_geo_join'] = df['NAME'].str[-11:]

# # Changes:

# # Low_Income_Marked -> LowIncome_Marked

# # df['Boundary_geo_join'] = df['Geography'].str[-11:] -> df['Boundary_geo_join'] = df['NAME'].str[-11:]

In [ ]:
# # Another way to do this is just take the last column before the ones we want, in this case; 'Year'

# year_index = df.columns.get_loc('Year')  # Get the index of the Year column
# prefixes = set(col.split('_')[0] for col in df.columns[year_index + 1:])

In [210]:
# #### KEEP


# # Define the categories as before
# categories = [
#     ('Highest Priority', ['Minority_identified', 'LowIncome_identified', 'Other_factors_identified', 'CES4_designated']),
#     ('LowInc/Min/Oth', ['Minority_identified', 'LowIncome_identified', 'Other_factors_identified']),
#     ('LowInc/Min/CES', ['Minority_identified', 'LowIncome_identified', 'CES4_designated']),
#     ('LowInc/Min', ['Minority_identified', 'LowIncome_identified']),
#     ('LowInc/Oth', ['LowIncome_identified', 'Other_factors_identified']),
#     ('Min/Oth', ['Minority_identified', 'Other_factors_identified']),
#     ('LowInc or Min and CES', ['LowIncome_identified', 'Minority_identified', 'CES4_designated']),  # Adjusted case
#     ('Minority', ['Minority_identified']),
#     ('Low Income', ['LowIncome_identified']),
#     ('Other factors', ['Other_factors_identified']),
#     ('CES4', ['CES4_designated'])
# ]

# # Create a copy of the DataFrame to avoid modifying the original DataFrame
# testy_copy = testy.copy()

# # Apply conditions and set corresponding columns to 1 where conditions are True
# for category, factors in categories:
#     if category == 'LowInc or Min and CES':
#         # Special case where 'LowIncome_identified' or 'Minority_identified' can be True
#         condition = ((testy_copy['LowIncome_identified'] == 1) | (testy_copy['Minority_identified'] == 1)) & (testy_copy['CES4_designated'] == 1)
#     else:
#         # General case: all factors must be 1
#         condition = np.all([testy_copy[factor] == 1 for factor in factors], axis=0)
    
#     # Assign 1 where the condition is True
#     testy_copy.loc[condition, category] = 1

# # Now we subset the dataframe to only keep the first six columns and the condition columns
# # Get the names of the first six columns
# first_six_columns = testy_copy.columns[:6]

# # Get the names of the category columns (the ones you added for the conditions)
# category_columns = [category[0] for category in categories]

# columns_to_keep = list(first_six_columns) + list('Geography', 'Geographic Area Name') + category_columns

# # Subset the dataframe to only keep these columns, making a copy to avoid the warning
# new_frame = testy_copy[columns_to_keep].copy()

# new_frame = new_frame.fillna(0)

# # Derive year from the input file name
# year_match = re.search(r'(\d{4})', Path(input_file).name)
# year = year_match.group(1) if year_match else 'UnknownYear'

In [235]:
# categories = [
#     ('Highest Priority', ['Minority_identified', 'LowIncome_identified', 'Other_factors_identified', 'CES4_designated']),
#     ('LowInc/Min/Oth', ['Minority_identified', 'LowIncome_identified', 'Other_factors_identified']),
#     ('LowInc/Min/CES', ['Minority_identified', 'LowIncome_identified', 'CES4_designated']),
#     ('LowInc/Min', ['Minority_identified', 'LowIncome_identified']),
#     ('LowInc/Oth', ['LowIncome_identified', 'Other_factors_identified']),
#     ('Min/Oth', ['Minority_identified', 'Other_factors_identified']),
#     ('LowInc or Min and CES', ['LowIncome_identified | Minority_identified', 'CES4_designated']),
#     ('Minority', ['Minority_identified']),
#     ('Low Income', ['LowIncome_identified']),
#     ('Other factors', ['Other_factors_identified']),
#     ('CES4', ['CES4_designated'])
# ]

# conditions = []
# for category in categories:
#     if category[0] == 'LowInc or Min and CES':
#         # Special case where 'LowIncome_identified' or 'Minority_identified' can be True
#         condition = ((testy['LowIncome_identified'] == 1) | (testy['Minority_identified'] == 1)) & (testy['CES4_designated'] == 1)
#     else:
#         # Standard case with 'and' logic across all columns in the category
#         condition = np.all([(testy[factor] == 1) for factor in category[1]], axis=0)
#     conditions.append(condition)

# # The labels (choices)
# choices = [category[0] for category in categories]

# # Assign the EJ_Label column based on the conditions
# testy['EJ_Label'] = np.select(conditions, choices, default='')

# # Derive year from the input file name
# year_match = re.search(r'(\d{4})', Path(input_file).name)
# year = year_match.group(1) if year_match else 'UnknownYear'

In [454]:
# import matplotlib.pyplot as plt 
# import plotly
# import plotly.graph_objects as go
# import plotly.express as px
# import plotly.io as pio

# fig = px.histogram(x = final_frame['LowIncome_Percent'])
# fig.show()